In [ ]:
import os
import math
import glob
import copy
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

import monai
from monai.data import Dataset, CacheDataset

from monai.networks.nets.resnet import ResNet, resnet18, resnet50, resnet101
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

from pytorch_lightning.loggers import MLFlowLogger, CSVLogger
from scipy.special import erf



torch.set_float32_matmul_precision("high")

DATA_ROOT = "data"
WORKING_DIR = os.getcwd()

# Encoder для 2.5D
ENCODER_NAME = 'efficientnet_b3.ra2_in1k' # 'convnext_tiny.fb_in22k', 'inception_v3.tf_in1k' "efficientnet_b0.ra_in1k" "efficientnet_b3.ra2_in1k" 
ENCODER_PRETRAINED = True
N_QUERIES = 64
NUM_HEADS = 8
INNER_DIM = 256

has_gpu = torch.cuda.is_available()
EXPERIMENT = "knee_mri_multilabel"
RUN_NAME = f"old_dataset_{ENCODER_NAME}_{N_QUERIES}q"

BATCH_SIZE = 8
ACCUMULATE = 2
NUM_EPOCHS = 12
LIMIT_TRAIN_BATCHES = 1.0
LIMIT_VAL_BATCHES = 1.0
PRECISION = "16-mixed"

# =========================================================
# 2.5D настройки
# =========================================================
SLICE_COUNT = 32
SLICE_FRACTIONS = np.arange(0.1, 0.9, 0.8/SLICE_COUNT).tolist()

SIZE = (384, 384)
SLICE_SIZE = (256, 256)


LABEL_COLUMNS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

CONFIG = dict(
    seed=42,
    data_root=DATA_ROOT,
    label_columns=LABEL_COLUMNS,
    batch_size=BATCH_SIZE,
    num_workers=12,
    epochs=NUM_EPOCHS,
    learning_rate=3e-4,
    weight_decay=1e-2,
    precision=PRECISION,
    gpus=1,
    limit_train_batches=LIMIT_TRAIN_BATCHES,
    limit_val_batches=LIMIT_VAL_BATCHES,
    accumulate_grad_batches=ACCUMULATE,

    # 2.5D
    slice_count=SLICE_COUNT,
    slice_fractions=SLICE_FRACTIONS,
    slice_size=SLICE_SIZE,
    encoder=dict(
        name=ENCODER_NAME,
        pretrained=ENCODER_PRETRAINED,
        n_queries=N_QUERIES,
        num_heads=NUM_HEADS,
        pooling_inner_dim=INNER_DIM
    ),
    encoder_lr_factor=1.0,  # encoder обучаем медленнее, head быстрее
    pseudo_csv="refined_predictions_perclass_v3.csv",

    # Послойный вес для BCE. По умолчанию выключен,
    # особенно если train-метки мягкие/вероятности.
    use_pos_weight=False,
    max_pos_weight=3.0,

    mlflow=dict(
        tracking_uri=os.path.join(WORKING_DIR, "mlruns"),
        experiment_name=EXPERIMENT,
        run_name=RUN_NAME,
    ),
)

print("DATA_ROOT =", DATA_ROOT)
print("WORKING_DIR =", WORKING_DIR)
print("CUDA =", has_gpu)
print("BATCH_SIZE =", BATCH_SIZE)
print("EPOCHS =", NUM_EPOCHS)
print("SLICE_FRACTIONS =", SLICE_FRACTIONS)
print("SLICE_SIZE =", SLICE_SIZE)
print("ENCODER =", ENCODER_NAME)


In [ ]:
import timm 
timm.list_pretrained()

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

from monai.data import MetaTensor
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Resized,
    RandRotated,
    RandFlipd,
    RandScaleIntensityd,
    EnsureTyped,
    Transform,
    NormalizeIntensityd,
    RandGaussianNoised,
    RandGaussianSmoothd,
    RandRotated,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandZoomd,
    RandAdjustContrastd,
    OneOf,
    RandSpatialCropd,
    CenterSpatialCropd
)

def get_train_transform(cfg: dict) -> Compose:
    return Compose(
        [
            Resized(keys=["image"], spatial_size=(-1, *SIZE)),
            #RandRotated(keys=["image"],prob=1.0, range_x=(-3.1415/12, 3.1415/12), padding_mode="constant"), #range_x=(0, 3.1415/12), range_y=(0, 3.1415/12), 
            RandZoomd(keys=["image"], prob=0.2, min_zoom=0.8, max_zoom=1.2, padding_mode="constant"),
            #RandSpatialCropd(keys=["image"], roi_size=(SLICE_COUNT, *SLICE_SIZE) ),
            CenterSpatialCropd(keys=["image"], roi_size=(-1, *SLICE_SIZE)),
            # Нормализуем весь исходный объем, как в baseline
            #RandFlipd(keys=["image"], prob=0.5, spatial_axis=[-1]),
            #RandGaussianNoised(keys=["image"], prob=0.1, mean=0.0, std=0.33),
            # RandGaussianSmoothd(
            #     keys=["image"],
            #     prob=0.1,
            #     sigma_x=(0.25, 1.5),
            #     sigma_y=(0.25, 1.5),
            #     sigma_z=(0.25, 1.5),
            # ),
            
            RandAdjustContrastd(keys=["image"], prob=0.1, gamma=(0.7, 1.5)),
            NormalizeIntensityd(keys=["image"]),

            # RandScaleIntensityd(keys=["image"], prob=0.1, factors=0.1),
            # RandShiftIntensityd(keys=["image"], prob=0.1, offsets=0.1),

        ]
    )


def get_val_transform(cfg: dict) -> Compose:
    return Compose(
        [
            Resized(keys=["image"], spatial_size=(-1, *SIZE)),
            CenterSpatialCropd(keys=["image"], roi_size=(-1, *SLICE_SIZE)),
            NormalizeIntensityd(keys=["image"]),
        ]
    )



In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import pydicom

import pytorch_lightning as pl
from torch.utils.data import DataLoader
from typing import Dict, List, Optional


class KneeOneSeriesSlicesDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        df_labels: pd.DataFrame,
        df_series: pd.DataFrame,
        label_columns: List[str],
        series_root: str,
        split: str,
        num_slices: int,
        slice_size: tuple,
        slice_fractions: List[float],
        transform=None,
    ):
        self.split = str(split)
        self.label_columns = list(label_columns)
        self.series_root = str(series_root)
        self.num_slices = int(num_slices)
        self.slice_size = tuple(int(x) for x in slice_size)
        self.slice_fractions = [float(x) for x in slice_fractions]
        self.transform = transform

        if len(self.slice_fractions) != self.num_slices:
            # На случай, если в конфиге передали другое количество
            if len(self.slice_fractions) < self.num_slices:
                self.slice_fractions = np.linspace(0.25, 0.75, self.num_slices).tolist()
            else:
                self.slice_fractions = self.slice_fractions[: self.num_slices]

        # ------------------------------------------------------------
        # Подготавливаем labels
        # ------------------------------------------------------------
        df_labels = df_labels.copy()
        df_labels["StudyInstanceUID"] = df_labels["StudyInstanceUID"].astype(str)

        for c in self.label_columns:
            if c not in df_labels.columns:
                raise KeyError(f"Label column {c} not found in df_labels")
            df_labels[c] = pd.to_numeric(df_labels[c], errors="coerce")

        df_labels = df_labels.dropna(subset=self.label_columns).copy()
        df_labels = df_labels.drop_duplicates(
            subset="StudyInstanceUID",
            keep="first",
        ).copy()

        # ------------------------------------------------------------
        # Подготавливаем series
        # ------------------------------------------------------------
        df_series = df_series.copy()
        df_series["StudyInstanceUID"] = df_series["StudyInstanceUID"].astype(str)
        df_series["SeriesInstanceUID"] = df_series["SeriesInstanceUID"].astype(str)

        df_series = df_series.drop_duplicates(
            subset=["StudyInstanceUID", "SeriesInstanceUID"],
            keep="first",
        ).copy()

        # ------------------------------------------------------------
        # Формируем все study * series
        # ------------------------------------------------------------
        merged = df_labels[
            ["StudyInstanceUID"] + self.label_columns
        ].merge(
            df_series[
                ["StudyInstanceUID", "SeriesInstanceUID"]
            ],
            on="StudyInstanceUID",
            how="inner",
        )

        merged = merged.drop_duplicates(
            subset=["StudyInstanceUID", "SeriesInstanceUID"],
            keep="first",
        )

        self.items: List[Dict] = []

        for _, row in merged.iterrows():
            study_uid = str(row["StudyInstanceUID"])
            series_uid = str(row["SeriesInstanceUID"])

            series_path = os.path.join(
                self.series_root,
                study_uid,
                series_uid,
            )

            if not os.path.isdir(series_path):
                continue

            # Пропускаем пустые папки
            try:
                has_files = any(
                    os.path.isfile(os.path.join(series_path, f))
                    for f in os.listdir(series_path)
                )
            except Exception:
                has_files = False

            if not has_files:
                continue

            label = row[self.label_columns].to_numpy(dtype=np.float32)

            self.items.append(
                {
                    "study_uid": study_uid,
                    "series_uid": series_uid,
                    "path": series_path,
                    "label": label,
                }
            )

        if not self.items:
            raise RuntimeError(
                f"[{self.split}] No valid study*series items found."
            )

    def __len__(self) -> int:
        return len(self.items)

    def _read_slice(self, path: str) -> torch.Tensor:
        """
        Читаем один DICOM-файл и возвращаем (H, W) tensor.
        """
        ds = pydicom.dcmread(path, force=True)
        img = np.asarray(ds.pixel_array, dtype=np.float32)
        img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

        # Приводим к 2D
        while img.ndim > 2:
            if 1 in img.shape:
                img = img.squeeze()
            else:
                img = img[0]

        if img.ndim == 1:
            img = img.reshape(1, -1)

        if img.ndim != 2:
            raise ValueError(
                f"Cannot convert DICOM slice to 2D: path={path}, shape={img.shape}"
            )

        t = torch.as_tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

        return t.squeeze().float().contiguous()

    def _list_files(self, series_path: str) -> List[str]:
        """
        Получает список файлов и сортирует их по DICOM тегу InstanceNumber.
        """
        files_raw = [
            f for f in os.listdir(series_path)
            if os.path.isfile(os.path.join(series_path, f))
        ]
        
        if not files_raw:
            return []
            
        # Создаем список кортежей (InstanceNumber, filename)
        file_info = []
        for f in files_raw:
            full_path = os.path.join(series_path, f)
            try:
                ds = pydicom.dcmread(full_path, stop_before_pixels=True)
                inst_num = getattr(ds, 'InstanceNumber', 0)
                if inst_num is None:
                    inst_num = 0
                file_info.append((int(inst_num), f))
            except Exception:
                # Если не удалось прочитать заголовок, ставим в конец или пропускаем
                # Для надежности лучше пропустить битые файлы на этапе листинга, 
                # но здесь просто присвоим большой номер, чтобы они ушли в конец, 
                # а потом отфильтруются при чтении
                file_info.append((999999, f))
        
        # Сортируем по InstanceNumber
        file_info.sort(key=lambda x: x[0])
        
        return [f[1] for f in file_info]

    def _pick_positions(self, n_files: int) -> List[int]:
        """
        Возвращает num_slices уникальных позиций в отсортированном порядке.
        
        На трейне выбираем случайно, на валидации — равномерно (как было раньше).
        """
        # На валидации оставляем старое поведение для стабильности метрик
        if self.split == "val":
            positions = []
            for frac in self.slice_fractions:
                pos = int(n_files * frac)
                pos = int(np.clip(pos, 0, n_files - 1))
                positions.append(pos)
            return sorted(positions)

        # --- Логика для TRAIN ---
        # Если файлов меньше или равно нужному количеству, берем все
        if n_files <= self.num_slices:
            return list(range(n_files))

        # Случайный выбор без замены (без дупликатов)
        random_indices = np.random.choice(
            n_files, 
            size=self.num_slices, 
            replace=False
        )
        
        # Сортируем, чтобы сохранить анатомический порядок (сверху вниз / снизу вверх)
        return sorted(random_indices.tolist())

    def __getitem__(self, idx: int) -> Dict:
        item = self.items[idx]

        study_uid = item["study_uid"]
        series_uid = item["series_uid"]
        series_path = item["path"]
        label = item["label"]

        files = self._list_files(series_path)

        if not files:
            raise RuntimeError(
                f"No files found in series folder: {series_path}"
            )

        n_files = len(files)

        positions = self._pick_positions(n_files)

        slices = []

        for pos in positions:
            loaded = False

            # Если нужный файл не читается, пробуем соседние позиции.
            # Это защита от битых/пробелных/не-DICOM файлов.
            for offset in range(n_files):
                cur_pos = (pos + offset) % n_files
                file_path = os.path.join(series_path, files[cur_pos])

                try:
                    sl = self._read_slice(file_path)
                    slices.append(sl)
                    loaded = True
                    break
                except Exception as e:
                    print(
                        f"[{self.split}] "
                        f"study={study_uid} series={series_uid} "
                        f"failed to read {file_path}: {e}"
                    )
                    continue

            if not loaded:
                continue

        if not slices:
            raise RuntimeError(
                f"Could not load any slices from series: {series_path}"
            )

        # Гарантируем точный num_slices
        while len(slices) < self.num_slices:
            slices.append(slices[-1].clone())

        if len(slices) > self.num_slices:
            slices = slices[: self.num_slices]

        # (num_slices, H, W)
        image = torch.stack(slices, dim=0).float().contiguous().unsqueeze(0)


        data = {
            "image": image,
            "label": torch.as_tensor(label, dtype=torch.float32),
        }

        # Обычные трансформы
        if self.transform is not None:
            data = self.transform(data)

        image = data["image"]

        if hasattr(image, "as_tensor"):
            image = image.as_tensor()



        image = torch.as_tensor(image, dtype=torch.float32).permute(1, 0, 2, 3)


        # MONAI трансформы дают (num_slices, H, W)
        # Модель хочет (num_slices, 1, H, W)
        if image.ndim == 3:
            image = image.unsqueeze(1)
        elif image.ndim == 4:
            if image.shape[0] != self.num_slices and image.shape[1] == self.num_slices:
                image = image.permute(1, 0, 2, 3)
        else:
            raise ValueError(
                f"Unexpected image ndim={image.ndim}, shape={image.shape}"
            )

        if image.shape[0] != self.num_slices:
            if image.numel() == self.num_slices * image.shape[-1] * image.shape[-2]:
                image = image.reshape(
                    self.num_slices,
                    1,
                    image.shape[-2],
                    image.shape[-1],
                )
            else:
                raise ValueError(
                    f"Expected image.shape[0]={self.num_slices}, got {image.shape}"
                )

        label = data["label"]

        if hasattr(label, "as_tensor"):
            label = label.as_tensor()

        label = torch.as_tensor(label, dtype=torch.float32).reshape(-1)

        return {
            "image": image.contiguous(),
            "label": label.contiguous(),
        }


class KneeOneSeriesSlicesDataModule(pl.LightningDataModule):
    """
    DataModule создает train/val датасеты по всем study*series.

    Никакой фильтрации series по plane/FS/axis здесь нет.
    """

    def __init__(self, cfg: dict):
        super().__init__()
        self.cfg = cfg
        self.train_ds = None
        self.val_ds = None

    def setup(self, stage: Optional[str] = None) -> None:
        cfg = self.cfg
        data_root = cfg["data_root"]
        label_columns = cfg["label_columns"]

        train_csv_path = os.path.join(data_root, "train.csv")
        pseudo_csv_path = os.path.join(data_root, cfg["pseudo_csv"])
        series_csv_path = os.path.join(data_root, "train_series.csv")
        series_root = os.path.join(data_root, "train_series")

        df_full = pd.read_csv(
            train_csv_path,
            usecols=["StudyInstanceUID"] + label_columns,
            dtype={"StudyInstanceUID": str},
        )

        df_pseudo = pd.read_csv(
            pseudo_csv_path,
            usecols=["StudyInstanceUID"] + label_columns,
            dtype={"StudyInstanceUID": str},
        )

        df_series = pd.read_csv(
            series_csv_path,
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        )

        # Val = настоящие labeled studies
        val_mask = df_full[label_columns].notna().any(axis=1)
        val_df = df_full[val_mask].copy()
        val_uids = set(val_df["StudyInstanceUID"].unique())

        # Train = pseudo labels, кроме val
        train_df = df_pseudo[
            ~df_pseudo["StudyInstanceUID"].isin(val_uids)
        ].copy()

        print(f"Val studies: {len(val_df)}, Train studies: {len(train_df)}")

        if len(train_df) == 0 or len(val_df) == 0:
            raise RuntimeError("Train or Val df is empty.")

        common = dict(
            df_series=df_series,
            label_columns=label_columns,
            series_root=series_root,
            num_slices=int(cfg["slice_count"]),
            slice_size=tuple(cfg["slice_size"]),
            slice_fractions=cfg["slice_fractions"],
        )

        self.train_ds = KneeOneSeriesSlicesDataset(
            df_labels=train_df,
            transform=get_train_transform(cfg),
            split="train",
            **common,
        )

        self.val_ds = KneeOneSeriesSlicesDataset(
            df_labels=val_df,
            transform=get_val_transform(cfg),
            split="val",
            **common,
        )

        print("Train dataset len:", len(self.train_ds))
        print("Val dataset len:", len(self.val_ds))

        if len(self.train_ds) == 0 or len(self.val_ds) == 0:
            raise RuntimeError("Dataset is empty after study*series expansion.")

    def _make_loader(self, dataset, shuffle: bool) -> DataLoader:
        cfg = self.cfg
        workers = int(cfg["num_workers"])

        loader_kwargs = {}
        if workers > 0:
            loader_kwargs["prefetch_factor"] = 2

        return DataLoader(
            dataset,
            batch_size=int(cfg["batch_size"]),
            shuffle=shuffle,
            num_workers=workers,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=workers > 0,
            drop_last=False,
            **loader_kwargs,
        )

    def train_dataloader(self) -> DataLoader:
        return self._make_loader(self.train_ds, shuffle=True)

    def val_dataloader(self) -> DataLoader:
        return self._make_loader(self.val_ds, shuffle=False)



In [ ]:
cfg = CONFIG
dm = KneeOneSeriesSlicesDataModule(cfg)
dm.setup()

tdl = dm.val_dataloader()



import matplotlib.pyplot as plt
import numpy as np

# %matplotlib inline

batch = next(iter(tdl))
images = batch["image"].detach().cpu().float()
labels = batch["label"].detach().cpu().numpy()

B, N_slices, C, H, W = images.shape

print("Batch image shape:", images.shape)
print(f"B={B}, N_slices={N_slices}, C={C}, H={H}, W={W}")
print("Labels shape:", labels.shape)
print("Labels sample 0:", labels[0])


def visualize_2p5d(images_one: torch.Tensor, title: str = "2.5D sample"):
    """
    images_one: (num_slices, 1, H, W)
    """
    N = images_one.shape[0]
    print(images_one.shape)

    fig, axes = plt.subplots(1, N, figsize=(4.2 * N, 4.2), squeeze=False)
    axes = np.atleast_1d(axes)

    for i in range(N):
        sl = images_one[i, 0].numpy()
        sl = np.nan_to_num(sl)

        p1, p99 = np.percentile(sl, (1, 99))
        if p99 <= p1:
            p1, p99 = 0, 1

        axes[0][i].imshow(
            sl,
            cmap="gray",
            vmin=p1,
            vmax=p99,
            origin="lower",
        )
        axes[0][i].set_title(f"Slice {i + 1}", fontsize=9)
        axes[0][i].axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


for b in range(min(4, B)):
    visualize_2p5d(images[b], title=f"Batch sample {b}")




In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import numpy as np
from typing import List, Optional, Dict

import timm
from sklearn.metrics import roc_auc_score


class GatedCrossAttentionPooling(nn.Module):
    def __init__(self, d_model: int = 768, n_queries: int = 16, num_heads: int = 8, 
                 gate_type: str = "elementwise", dropout: float = 0.1,
                 inner_dim: Optional[int] = None):  # <-- новый параметр
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.n_queries = n_queries
        self.num_heads = num_heads
        self.gate_type = gate_type
        
        # Если inner_dim не указан, используем d_model
        self.inner_dim = inner_dim if inner_dim is not None else d_model
        assert self.inner_dim % num_heads == 0, f"inner_dim ({self.inner_dim}) must be divisible by num_heads ({num_heads})"
        self.head_dim = self.inner_dim // num_heads

        # Learnable queries
        self.query_tokens = nn.Parameter(torch.zeros(1, n_queries, self.inner_dim))
        nn.init.trunc_normal_(self.query_tokens, std=0.02)
        
        # Проекция во внутреннее пространство
  
        self.proj_k = nn.Linear(d_model, self.inner_dim)
        self.proj_v = nn.Linear(d_model, self.inner_dim)

        # Gate projection
        if gate_type == "headwise":
            self.proj_gate = nn.Linear(self.inner_dim, num_heads)
        else:
            self.proj_gate = nn.Linear(self.inner_dim, self.inner_dim)
            
        self.norm_q = nn.LayerNorm(self.inner_dim)
        self.norm_kv = nn.LayerNorm(d_model)
        self.norm_out = nn.LayerNorm(self.inner_dim)
        self.dropout = nn.Dropout(dropout)
        self.proj_out = nn.Linear(self.inner_dim, self.inner_dim)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, *spatial = x.shape
        L = math.prod(spatial)

        tokens = x.flatten(2).transpose(1, 2)  # (B, L, C)
        
        tokens_norm = self.norm_kv(tokens)
        k = self.proj_k(tokens_norm)
        v = self.proj_v(tokens_norm)

        q = self.query_tokens.expand(B, -1, -1)
        q_norm = self.norm_q(q)

        q_attn = q_norm.view(B, self.n_queries, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        out = F.scaled_dot_product_attention(q_attn, k, v)
        out = out.transpose(1, 2).reshape(B, self.n_queries, self.inner_dim)

        #Gating
        if self.gate_type == "headwise":
            gate = torch.sigmoid(self.proj_gate(q_norm))
            gate = gate.unsqueeze(-1)
            out_reshaped = out.view(B, self.n_queries, self.num_heads, self.head_dim)
            out = (out_reshaped * gate).reshape(B, self.n_queries, self.inner_dim)
        else:
            gate = torch.sigmoid(self.proj_gate(q_norm))
            out = out * gate
        
        out = self.norm_out(out)
        out = self.dropout(out)
        out = self.proj_out(out)

        return out.flatten(1)

class KneeConvNeXtTwoHalfDMultilabel(pl.LightningModule):
    def __init__(
        self,
        cfg: dict,
        label_names: List[str],
        encoder_name: str = "convnext_tiny.fb_in1k",
        pretrained: bool = True,
        n_queries: int = 16,
        num_heads: int = 8,
        pooling_inner_dim: int = 128, 
        pos_weight: Optional[torch.Tensor] = None,
    ):
        super().__init__()

        self.cfg = cfg
        self.label_names = list(label_names)
        self.num_classes = len(self.label_names)

        self.encoder = timm.create_model(
            encoder_name,
            pretrained=pretrained,
            in_chans=1,
            num_classes=0,
            global_pool="",
        )

        # model = timm.create_model('resnet50', pretrained=False, num_classes=1000)
        # model.fc = nn.Identity()
        # model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

        # # Загружаем веса
        # state_dict = torch.load('weights/converted_resnet50.pt', weights_only=False)
        # model.load_state_dict(state_dict['model_state_dict'])
        # model.global_pool = nn.Identity()

        # self.encoder = model

        self.d_model = int(self.encoder.num_features)
        self.norm = nn.BatchNorm2d(self.d_model)

        self.cap = GatedCrossAttentionPooling(
            d_model=self.d_model,
            n_queries=n_queries,
            num_heads=num_heads,
            gate_type="headwise",
            inner_dim=pooling_inner_dim  # <-- например, 256 вместо 768
        )

        # Classifier теперь работает с уменьшенной размерностью
        self.classifier = nn.Linear(n_queries * pooling_inner_dim if pooling_inner_dim else n_queries * self.d_model, self.num_classes)

        if pos_weight is not None:
            if not isinstance(pos_weight, torch.Tensor):
                pos_weight = torch.as_tensor(pos_weight, dtype=torch.float32)
            self.register_buffer("pos_weight", pos_weight, persistent=False)
        else:
            self.pos_weight = None

        self.val_preds: List[torch.Tensor] = []
        self.val_labels: List[torch.Tensor] = []

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N_slices, C, H, W = x.shape
        x_flat = x.reshape(B * N_slices, C, H, W)

        feats = self.encoder(x_flat)
        feats = self.norm(feats)
        
        Cf, Hf, Wf = feats.shape[1:]

        feats = feats
        
        feats_3d = feats.view(B, N_slices, Cf, Hf, Wf).permute(0, 2, 1, 3, 4).contiguous()
        pooled = self.cap(feats_3d)

        logits = self.classifier(pooled)
        return logits

    def _loss(self, logits: torch.Tensor, labels: torch.Tensor, train=True) -> torch.Tensor:
        # 1. Инициализируем функцию потерь с reduction='none', чтобы получить лосс для каждого элемента
        if self.pos_weight is not None and train:
            pos_weight = self.pos_weight.to(logits.device).float()
            return nn.BCEWithLogitsLoss(pos_weight=pos_weight)(logits, labels)
        else:
            return nn.BCEWithLogitsLoss()(logits, labels)

    def training_step(self, batch: Dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        images = batch["image"]
        labels = batch["label"]

        logits = self(images)
        loss = self._loss(logits, labels)

        self.log(
            "train_loss",
            loss,
            prog_bar=True,
            on_step=True,
            on_epoch=True,
            batch_size=images.size(0),
            sync_dist=True,
        )

        return loss

    def validation_step(self, batch: Dict[str, torch.Tensor], batch_idx: int) -> None:
        images = batch["image"]
        labels = batch["label"]

        logits = self(images)
        loss = self._loss(logits, labels, train=False)

        self.log(
            "val_loss",
            loss,
            prog_bar=True,
            on_step=True,
            on_epoch=True,
            batch_size=images.size(0),
            sync_dist=True,
        )

        probs = torch.sigmoid(logits)

        self.val_preds.append(probs.detach().cpu())
        self.val_labels.append(labels.detach().cpu())

        return loss

    def _safe_binary_auc(
        self,
        y_true: np.ndarray,
        y_score: np.ndarray,
        threshold: float = 0.5,
        allow_soft_labels: bool = True,
    ) -> Optional[float]:
        mask = np.isfinite(y_true) & np.isfinite(y_score)
        if int(mask.sum()) < 2:
            return None

        yt = y_true[mask]
        yp = y_score[mask]

        valid = (yt >= 0.0) & (yt <= 1.0)
        yt = yt[valid]
        yp = yp[valid]

        if len(yt) < 2:
            return None

        yt_round = np.round(yt, 6)
        unique_vals = np.unique(yt_round)

        if np.all(np.isin(unique_vals, [0.0, 1.0])):
            yt_bin = yt_round.astype(int)
        else:
            if not allow_soft_labels:
                return None
            yt_bin = (yt > threshold).astype(int)

        if len(np.unique(yt_bin)) < 2:
            return None

        if np.all(yp == yp[0]):
            return None

        try:
            return float(roc_auc_score(yt_bin, yp))
        except Exception:
            return None

    def _macro_auc(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        default_auc: Optional[float] = 0.5,
    ) -> float:
        if y_true.ndim == 1:
            y_true = y_true[:, None]
        if y_pred.ndim == 1:
            y_pred = y_pred[:, None]

        aucs = []

        for i in range(y_true.shape[1]):
            auc = self._safe_binary_auc(
                y_true[:, i],
                y_pred[:, i],
            )

            if auc is None:
                if default_auc is None:
                    continue
                auc = default_auc

            aucs.append(auc)

        if len(aucs) == 0:
            if default_auc is None:
                raise RuntimeError(
                    "Cannot compute macro AUC: no valid label has enough binary samples."
                )
            return float(default_auc)

        return float(np.mean(aucs))

    def on_validation_epoch_end(self) -> None:
        if len(self.val_preds) == 0:
            return

        y_pred = torch.cat(self.val_preds, dim=0).numpy()
        y_true = torch.cat(self.val_labels, dim=0).numpy()

        auc = self._macro_auc(y_true, y_pred, default_auc=0.5)
        self.log("val_auc", auc, prog_bar=True, on_epoch=True, sync_dist=True)

        for i, name in enumerate(self.label_names):
            auc_i = self._safe_binary_auc(
                y_true[:, i],
                y_pred[:, i],
            )

            if auc_i is None:
                continue

            safe_name = name.replace(" ", "_").replace("'", "_")
            self.log(
                f"val_auc_{safe_name}",
                float(auc_i),
                on_epoch=True,
            )

        self.val_preds.clear()
        self.val_labels.clear()

    def configure_optimizers(self):
        cfg = self.cfg

        base_lr = float(cfg["learning_rate"])
        weight_decay = float(cfg["weight_decay"])
        encoder_lr_factor = float(cfg["encoder_lr_factor"])

        encoder_params = list(self.encoder.parameters())
        encoder_param_ids = {id(p) for p in encoder_params}

        head_params = [
            p
            for p in self.parameters()
            if p.requires_grad and id(p) not in encoder_param_ids
        ]

        param_groups = []

        if encoder_params:
            param_groups.append(
                {
                    "params": encoder_params,
                    "lr": base_lr * encoder_lr_factor,
                }
            )

        if head_params:
            param_groups.append(
                {
                    "params": head_params,
                    "lr": base_lr,
                }
            )

        if not param_groups:
            raise RuntimeError("No trainable parameters found.")

        optimizer = torch.optim.AdamW(
            param_groups,
            lr=base_lr,
            weight_decay=weight_decay,
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(1, int(cfg["epochs"])),
            eta_min=1e-6,
        )

        return [optimizer], [scheduler]


In [ ]:
model = KneeConvNeXtTwoHalfDMultilabel(cfg, LABEL_COLUMNS)
out = model(torch.randn(2, 16, 1, 256, 256))
out.shape

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
from typing import Optional

from pytorch_lightning.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
)
from pytorch_lightning.loggers import MLFlowLogger, CSVLogger


def compute_pos_weight(
    train_df: pd.DataFrame,
    label_names: List[str],
    max_weight: float = 8.0,
    threshold: float = 0.5,
) -> Optional[torch.Tensor]:
    labels = train_df[label_names].apply(
        lambda col: pd.to_numeric(col, errors="coerce")
    )

    mask = ~labels.isna().any(axis=1)
    labels = labels[mask].to_numpy(dtype=np.float32)

    if len(labels) == 0:
        return None

    # Если метки мягкие, считаем positive как > threshold
    pos = (labels > threshold).sum(axis=0).astype(np.float32)
    neg = len(labels) - pos

    pos = np.clip(pos, 1.0, None)
    weights = neg / pos
    weights = np.clip(weights, 1.0, float(max_weight))

    return torch.tensor(weights, dtype=torch.float32)


def create_logger(cfg: dict):
    mlflow_cfg = cfg["mlflow"]

    try:
        logger = MLFlowLogger(
            tracking_uri=mlflow_cfg["tracking_uri"],
            experiment_name=mlflow_cfg["experiment_name"],
            run_name=mlflow_cfg["run_name"],
        )
        return logger
    except Exception as e:
        print(f"MLFlowLogger failed: {e}. Falling back to CSVLogger.")

    return CSVLogger(save_dir=os.path.join(WORKING_DIR, "lightning_logs"))


def train_model(
    cfg: Optional[dict] = None,
    limit_train_batches: Optional[float] = None,
    limit_val_batches: Optional[float] = None,
    max_epochs: Optional[int] = None,
):
    if cfg is None:
        cfg = CONFIG

    cfg = copy.deepcopy(cfg)

    pl.seed_everything(int(cfg["seed"]), workers=True)

    label_names = cfg["label_columns"]
    enc_cfg = cfg["encoder"]

    pos_weight = None

    if cfg["use_pos_weight"]:
        pseudo_csv_path = os.path.join(
            cfg["data_root"],
            cfg["pseudo_csv"],
        )

        try:
            df_pseudo = pd.read_csv(
                pseudo_csv_path,
                usecols=["StudyInstanceUID"] + label_names,
                dtype={"StudyInstanceUID": str},
            )

            pos_weight = compute_pos_weight(
                df_pseudo,
                label_names,
                max_weight=float(cfg["max_pos_weight"]),
            )

            if pos_weight is None:
                pos_weight = torch.ones(len(label_names), dtype=torch.float32)

        except Exception as e:
            print(f"Failed to compute pos_weight: {e}")
            pos_weight = None

    print(f"Pos weight: {pos_weight}")

    dm = KneeOneSeriesSlicesDataModule(cfg)

    model = KneeConvNeXtTwoHalfDMultilabel(
        cfg=cfg,
        label_names=label_names,
        encoder_name=str(enc_cfg["name"]),
        pretrained=bool(enc_cfg["pretrained"]),
        n_queries=int(enc_cfg["n_queries"]),
        num_heads=int(enc_cfg["num_heads"]),
        pooling_inner_dim=int(enc_cfg["pooling_inner_dim"]),
        pos_weight=pos_weight,
    )

    # model = KneeConvNeXtTwoHalfDMultilabel.load_from_checkpoint("checkpoints/best_epoch=3-val_auc=0.8476.ckpt", 
    #     cfg=cfg,
    #     label_names=label_names,
    #     encoder_name=str(enc_cfg["name"]),
    #     pretrained=bool(enc_cfg["pretrained"]),
    #     n_queries=int(enc_cfg["n_queries"]),
    #     num_heads=int(enc_cfg["num_heads"]),
    #     pos_weight=pos_weight,
    # )

    logger = create_logger(cfg)

    hyper = {
        k: str(v)
        for k, v in cfg.items()
        if isinstance(v, (str, int, float, bool))
    }

    hyper["encoder_name"] = str(enc_cfg["name"])
    hyper["encoder_pretrained"] = str(bool(enc_cfg["pretrained"]))
    hyper["n_queries"] = str(int(enc_cfg["n_queries"]))
    hyper["num_heads"] = str(int(enc_cfg["num_heads"]))
    hyper["slice_fractions"] = str(cfg["slice_fractions"])
    hyper["slice_size"] = str(tuple(cfg["slice_size"]))

    try:
        logger.log_hyperparams(hyper)
    except Exception as e:
        print(f"Failed to log hyperparams: {e}")

    checkpoint_dir = os.path.join(WORKING_DIR, "checkpoints")

    chpt = ModelCheckpoint(
                dirpath=checkpoint_dir,
                filename="best_{epoch}-{val_auc:.4f}",
                monitor="val_auc",
                mode="max",
                save_top_k=1,
                verbose=True
            )

    callbacks = [
        chpt, 
        LearningRateMonitor(logging_interval="epoch"),
        
        ModelCheckpoint(
            dirpath=checkpoint_dir,
            filename="best_{epoch}-{val_loss:.4f}",
            monitor="val_loss",
            mode="min",
            save_top_k=1,
        ),

    ]

    accelerator = "gpu" if torch.cuda.is_available() else "cpu"
    precision = str(cfg["precision"])

    if accelerator == "cpu":
        precision = "32"

    devices = int(cfg["gpus"]) if accelerator == "gpu" else 1

    if limit_train_batches is None:
        limit_train_batches = float(cfg["limit_train_batches"])

    if limit_val_batches is None:
        limit_val_batches = float(cfg["limit_val_batches"])

    trainer = pl.Trainer(
        max_epochs=cfg["epochs"],
        accelerator=accelerator,
        devices=devices,
        precision=precision,
        logger=[logger],
        callbacks=callbacks,
        val_check_interval=500,
        num_sanity_val_steps=2,
        limit_train_batches=limit_train_batches,
        limit_val_batches=limit_val_batches,
        accumulate_grad_batches=cfg["accumulate_grad_batches"],
        gradient_clip_val=1.0,
        enable_progress_bar=True,
    )

    trainer.fit(model, datamodule=dm)
    #trainer.fit(datamodule=dm, ckpt_path=)

    best_path = chpt.best_model_path
    best_metric = chpt.best_model_score
    
    print("Best checkpoint:", best_path)
    return {
        "best": best_path,
        "metric": best_metric,
        "data": cfg["pseudo_csv"]
    }


In [ ]:

TRAIN_RESULT = train_model()


In [26]:


class KneeModel(pl.LightningModule):
    def __init__(
        self,
        encoder_name: str = "convnext_tiny.fb_in1k",
        n_queries: int = N_QUERIES,
        num_heads: int = NUM_HEADS,
        pooling_inner_dim: int = INNER_DIM
    ):
        super().__init__()


        self.encoder = timm.create_model(
            encoder_name,
            pretrained=False,
            in_chans=1,
            num_classes=0,
            global_pool="",
        )

        self.d_model = int(self.encoder.num_features)

        self.norm = nn.BatchNorm2d(self.d_model)

        self.cap = GatedCrossAttentionPooling(
            d_model=self.d_model,
            n_queries=n_queries,
            num_heads=num_heads,
            gate_type="headwise",
            inner_dim=pooling_inner_dim  # <-- например, 256 вместо 768
        )
        
        # Classifier теперь работает с уменьшенной размерностью
        self.classifier = nn.Linear(n_queries * pooling_inner_dim if pooling_inner_dim else n_queries * self.d_model, len(LABEL_COLUMNS))

    def forward(self, x1: torch.Tensor, x2: torch.Tensor, x3: torch.Tensor) -> torch.Tensor:
        x = torch.cat([x1, x2, x3], dim=1)
        B, N_slices, C, H, W = x.shape
        x_flat = x.reshape(B * N_slices, C, H, W)

        feats = self.encoder(x_flat)
        feats = self.norm(feats)
        #feats = self.do1(feats)

        Cf, Hf, Wf = feats.shape[1:]
        
        feats_3d = feats.view(B, N_slices, Cf, Hf, Wf).permute(0, 2, 1, 3, 4).contiguous()
        pooled = self.cap(feats_3d)

        logits = self.classifier(pooled)
        return logits



In [27]:

class Backbone(pl.LightningModule):
    def __init__(
        self,
        encoder_name: str = "convnext_tiny.fb_in1k",

    ):
        super().__init__()

        self.encoder = timm.create_model(
            encoder_name,
            pretrained=False,
            in_chans=1,
            num_classes=0,
            global_pool="",
        )

        self.d_model = int(self.encoder.num_features)
        self.norm = nn.BatchNorm2d(self.d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        B, N_slices, C, H, W = x.shape
        x_flat = x.reshape(B * N_slices, C, H, W)

        feats = self.encoder(x_flat)
        feats = self.norm(feats)
        #feats = self.do1(feats)

        Cf, Hf, Wf = feats.shape[1:]
        
        feats_3d = feats.view(B, N_slices, Cf, Hf, Wf).permute(0, 2, 1, 3, 4).contiguous()

        return feats_3d

class Head(pl.LightningModule):
    def __init__(
        self,
        feature_dim,
        n_queries: int = N_QUERIES,
        num_heads: int = NUM_HEADS,
        pooling_inner_dim: int = INNER_DIM
    ):
        super().__init__()

        self.d_model=feature_dim
        self.cap = GatedCrossAttentionPooling(
            d_model=self.d_model,
            n_queries=n_queries,
            num_heads=num_heads,
            gate_type="headwise",
            inner_dim=pooling_inner_dim  # <-- например, 256 вместо 768
        )
        
        # Classifier теперь работает с уменьшенной размерностью
        self.classifier = nn.Linear(n_queries * pooling_inner_dim if pooling_inner_dim else n_queries * self.d_model, len(LABEL_COLUMNS))

    def forward(self, feats_3d):
        pooled = self.cap(feats_3d)

        return self.classifier(pooled)



In [ ]:
BEST_CHECKPOINT = TRAIN_RESULT.get("best") 
print("Checkpoint to use:", BEST_CHECKPOINT)

model = KneeModel.load_from_checkpoint(BEST_CHECKPOINT, encoder_name=ENCODER_NAME, n_queries=N_QUERIES, num_heads=NUM_HEADS)
#model.to_torchscript(f"models/{TRAIN_RESULT["metric"]:.04f}_{TRAIN_RESULT["data"][:-4]}_{ENCODER_NAME}.torchscript", example_inputs=(torch.randn(2, 16, 1, 256, 256), torch.randn(2, 16, 1, 256, 256), torch.randn(2, 16, 1, 256, 256)), method="trace")
#model = KneeModel.load_from_checkpoint("checkpoints/best_epoch=11-val_auc=0.9092.ckpt", encoder_name="efficientnet_b3.ra2_in1k" , n_queries=N_QUERIES, num_heads=NUM_HEADS, map_location="cpu")
device = next(model.parameters()).device

backbone = Backbone(encoder_name="efficientnet_b3.ra2_in1k" ).to(device)
backbone.encoder = model.encoder
backbone.norm = model.norm
d_model = model.d_model
head= Head(d_model, n_queries=N_QUERIES, num_heads=NUM_HEADS).to(device)
head.cap = model.cap
head.classifier = model.classifier



backbone.to_torchscript(f"models/backbone_{TRAIN_RESULT["metric"]:.04f}_{TRAIN_RESULT["data"][:-4]}_{ENCODER_NAME}.torchscript", example_inputs=torch.randn(2, 16, 1, 256, 256), method="trace")
head.to_torchscript(f"models/head_{TRAIN_RESULT["metric"]:.04f}_{TRAIN_RESULT["data"][:-4]}_{ENCODER_NAME}.torchscript", example_inputs=torch.randn(2, d_model, 16, 8, 8), method="trace")

In [28]:


model = KneeModel.load_from_checkpoint("weights/best_epoch6-val_auc0.9070.ckpt", encoder_name=ENCODER_NAME, n_queries=N_QUERIES, num_heads=NUM_HEADS, pooling_inner_dim=1536)
#model.to_torchscript(f"models/{TRAIN_RESULT["metric"]:.04f}_{TRAIN_RESULT["data"][:-4]}_{ENCODER_NAME}.torchscript", example_inputs=(torch.randn(2, 16, 1, 256, 256), torch.randn(2, 16, 1, 256, 256), torch.randn(2, 16, 1, 256, 256)), method="trace")
#model = KneeModel.load_from_checkpoint("checkpoints/best_epoch=11-val_auc=0.9092.ckpt", encoder_name="efficientnet_b3.ra2_in1k" , n_queries=N_QUERIES, num_heads=NUM_HEADS, map_location="cpu")
device = next(model.parameters()).device

backbone = Backbone(encoder_name="efficientnet_b3.ra2_in1k" ).to(device)
backbone.encoder = model.encoder
backbone.norm = model.norm
d_model = model.d_model
head= Head(d_model, n_queries=N_QUERIES, num_heads=NUM_HEADS, pooling_inner_dim=1536).to(device)
head.cap = model.cap
head.classifier = model.classifier



backbone.to_torchscript(f"models/backbone_0.907.torchscript", example_inputs=torch.randn(2, 16, 1, 256, 256), method="trace")
head.to_torchscript(f"models/head_0.907.torchscript", example_inputs=torch.randn(2, d_model, 16, 8, 8), method="trace")

`LightningModule.to_torchscript` has been deprecated in v2.7 and will be removed in v2.8. TorchScript is deprecated in PyTorch. Use `torch.export.export()` for model exporting instead. See https://pytorch.org/docs/stable/export.html for more information.


Head(
  original_name=Head
  (cap): GatedCrossAttentionPooling(
    original_name=GatedCrossAttentionPooling
    (proj_k): Linear(original_name=Linear)
    (proj_v): Linear(original_name=Linear)
    (proj_gate): Linear(original_name=Linear)
    (norm_q): LayerNorm(original_name=LayerNorm)
    (norm_kv): LayerNorm(original_name=LayerNorm)
    (norm_out): LayerNorm(original_name=LayerNorm)
    (dropout): Dropout(original_name=Dropout)
    (proj_out): Linear(original_name=Linear)
  )
  (classifier): Linear(original_name=Linear)
)

In [ ]:
model = timm.create_model('efficientnet_b3.ra2_in1k')
print(model)

import torch
import torch.nn as nn

class EfficientNetMultiScale(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
        # Определяем индексы блоков для разных скейлов
        # Обычно берутся блоки с разным разрешением
        self.scale_indices = [0, 1, 2, 3, 4, 5, 6]  # все блоки
        
    def forward(self, x):
        features = {}
        
        # Начальный слой
        x = self.model.conv_stem(x)
        x = self.model.bn1(x)
        
        # Проходим по блокам и сохраняем выходы
        for i, block in enumerate(self.model.blocks):
            x = block(x)
            features[f'scale_{i}'] = x
            
        # Финальные слои
        x = self.model.conv_head(x)
        x = self.model.bn2(x)
        x = self.model.global_pool(x)
        x = self.model.classifier(x)
        
        return x, features

input_tensor = torch.randn((1, 3, 256, 256))

# Использование
multi_scale_model = EfficientNetMultiScale(model)
output, features = multi_scale_model(input_tensor)

for f in features:
    print(features[f].shape)
# features содержит фичамапы разных размеров:
# scale_0: высокий resolution, низкий semantic level
# scale_6: низкий resolution, высокий semantic level